In [112]:
import pandas as pd
import numpy as np
import os
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sn

In [113]:
# 1. Load dataset
df = pd.read_csv('diabetes_dataset.csv')
print("Initial data shape:", df.shape)
print(df.head());

Initial data shape: (100000, 16)
   year  gender   age location  race:AfricanAmerican  race:Asian  \
0  2020  Female  32.0  Alabama                     0           0   
1  2015  Female  29.0  Alabama                     0           1   
2  2015    Male  18.0  Alabama                     0           0   
3  2015    Male  41.0  Alabama                     0           0   
4  2016  Female  52.0  Alabama                     1           0   

   race:Caucasian  race:Hispanic  race:Other  hypertension  heart_disease  \
0               0              0           1             0              0   
1               0              0           0             0              0   
2               0              0           1             0              0   
3               1              0           0             0              0   
4               0              0           0             0              0   

  smoking_history    bmi  hbA1c_level  blood_glucose_level  diabetes  
0           never  27.32

In [114]:
# 2. Drop unneeded columns
df.drop(columns=[
    'year','location',
    'race:AfricanAmerican','race:Asian','race:Caucasian','race:Hispanic','race:Other','smoking_history'
], inplace=True)
print("After dropping columns:", df.shape)
print(df.head());

After dropping columns: (100000, 8)
   gender   age  hypertension  heart_disease    bmi  hbA1c_level  \
0  Female  32.0             0              0  27.32          5.0   
1  Female  29.0             0              0  19.95          5.0   
2    Male  18.0             0              0  23.76          4.8   
3    Male  41.0             0              0  27.32          4.0   
4  Female  52.0             0              0  23.75          6.5   

   blood_glucose_level  diabetes  
0                  100         0  
1                   90         0  
2                  160         0  
3                  159         0  
4                   90         0  


In [115]:
# 3. Encode gender
encoder = LabelEncoder()
df['gender'] = encoder.fit_transform(df['gender'])
print("Label Encoding Mapping:", dict(zip(encoder.classes_, range(len(encoder.classes_)))))

Label Encoding Mapping: {'Female': 0, 'Male': 1, 'Other': 2}


In [116]:
# 4. Drop duplicates & 'Other' gender
df.drop_duplicates(inplace=True)
df = df[df['gender'] != 2]
print("After dropping duplicates & 'Other':", df.shape)

After dropping duplicates & 'Other': (91295, 8)


In [117]:
# 5. Normalize numeric columns
cols_to_normalize = ['age','bmi','hbA1c_level','blood_glucose_level']
scaler = MinMaxScaler()
df[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])

In [118]:
# 6. Create binary thresholds
df['X1'] = (df['gender'] == 1).astype(int)  # 1 if female, 0 if male
df['X2'] = (df['age'] >= 60).astype(int)  # 1 if age >= 60, 0 if age < 60
df['X3'] = (df['bmi'] >= 23).astype(int)  # 1 if BMI >= 23, 0 if BMI < 23
df['X4'] = (df['blood_glucose_level'] >= 126/df['blood_glucose_level'].max()).astype(int)  # 1 if BG >= 126, 0 otherwise
df['X5'] = (df['hbA1c_level'] >= 6.5/df['hbA1c_level'].max()).astype(int)  # 1 if HbA1c >= 6.5, 0 otherwise

In [119]:
# 7. Generate all pairwise interaction binaries for X3 (BMI) with other features
for feature in ['X1', 'X2', 'X4', 'X5']:
    df[f'X3({df["X3"].unique()[0]})_{feature}({df[feature].unique()[0]})'] = (
        (df['X3'] == 0) & (df[feature] == 0)).astype(int)  # Create the interaction term and set 1 if both conditions hold

In [120]:
# 8. Drop raw binary columns
df.drop(columns=['X1','X2','X3','X4','X5'], inplace=True)
print("After creating interactions, total columns:", df.shape[1])

After creating interactions, total columns: 12


In [130]:
# 9. Select top 10 interaction features by mutual information
interaction_cols = [c for c in df.columns if '(' in c and '_' in c]
y = df['diabetes']
mi_scores = mutual_info_classif(df[interaction_cols], y, discrete_features=True)
mi_df = pd.DataFrame({'feature': interaction_cols, 'mi': mi_scores})
top_10 = mi_df.sort_values(by='mi', ascending=False).head(10)['feature'].tolist()
print("Top 10 interaction features:", top_10)

Top 10 interaction features: ['X3(0)_X1(0)', 'X3(0)_X2(0)', 'X3(0)_X4(0)', 'X3(0)_X5(0)']


In [132]:
# 10. Build final feature set (original + top 10 interactions)
X = pd.concat([df[['gender','age','hypertension','heart_disease','bmi','hbA1c_level','blood_glucose_level']], 
               df[top_10]], axis=1)
y = df['diabetes']

In [134]:
# 11. Split & SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [136]:
# 12. Train KNN with k=13
knn = KNeighborsClassifier(n_neighbors=13, metric='manhattan', weights='distance')
knn.fit(X_train_sm, y_train_sm)

KNeighborsClassifier(metric='manhattan', n_neighbors=13, weights='distance')

In [138]:
# 13. Predict and evaluate
y_pred = knn.predict(X_test)

In [140]:
print("\n=== KNN WITH INTERACTIONS (k=13) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


=== KNN WITH INTERACTIONS (k=13) ===
Accuracy: 0.9153841940960622
Precision: 0.5274181264280274
Recall: 0.8200118413262285
F1-score: 0.641946697566628

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.93      0.95     16570
           1       0.53      0.82      0.64      1689

    accuracy                           0.92     18259
   macro avg       0.75      0.87      0.80     18259
weighted avg       0.94      0.92      0.92     18259

Confusion Matrix:
 [[15329  1241]
 [  304  1385]]


In [142]:
print("AUC:", roc_auc_score(y_test, knn.predict_proba(X_test)[:, 1]))

AUC: 0.9396951876835915
